In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "The_Courage_to_be_Disliked.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} Pages from the PDF.")
print("\n--- First Page previe (first 500 chars) ---")
print(pages[2].page_content[:500])

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2660\196758495.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
D:\Workspace\agentic-ai-course-projects\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 212 Pages from the PDF.

--- First Page previe (first 500 chars) ---



In [3]:
print(pages[3].page_content[:500])

Contents
Authors’ Note
Introduction
THE FIRST NIGHT:
Deny Trauma
The Unknown Third Giant
Why People Can Change
Trauma Does Not Exist
People Fabricate Anger
How to Live Without Being Controlled by the Past
Socrates and Adler
Are You Okay Just As You Are?
Unhappiness Is Something You Choose for Yourself
People Always Choose Not to Change
Your Life Is Decided Here and Now
THE SECOND NIGHT:
All Problems Are Interpersonal Relationship Problems
Why You Dislike Yourself
All Problems Are Interpersonal R


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      # ~150 words per chunk
    chunk_overlap=100,   # overlap keeps context at boundaries
    separators=["\n\n", "\n", ".", " "], # tries paragraph -> line -> sentence -> word
)

chunks = splitter.split_documents(pages)

len(chunks)


796

In [5]:
chunks[1].page_content

'Contents\nAuthors’ Note\nIntroduction\nTHE FIRST NIGHT:\nDeny Trauma\nThe Unknown Third Giant\nWhy People Can Change\nTrauma Does Not Exist\nPeople Fabricate Anger\nHow to Live Without Being Controlled by the Past\nSocrates and Adler\nAre You Okay Just As You Are?\nUnhappiness Is Something You Choose for Yourself\nPeople Always Choose Not to Change\nYour Life Is Decided Here and Now\nTHE SECOND NIGHT:\nAll Problems Are Interpersonal Relationship Problems\nWhy You Dislike Yourself\nAll Problems Are Interpersonal Relationship Problems\nFeelings of Inferiority Are Subjective Assumptions'

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

PERSIST_DIR = "./chroma_db"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(PERSIST_DIR):
    # Already built — just load it, don't re-embed
    vector_store = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embeddings,
    )
    print("Loaded existing vector store.")
else:
    # First run — build it and save to disk
    vector_store = Chroma.from_documents(
        chunks,
        embeddings,
        persist_directory=PERSIST_DIR,
    )
    print("Built and persisted new vector store.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5095.94it/s]


In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

test_query = "Who is the third major figure in psychology alongside Freud and Jung?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
YOUTH: Individual psychology? Another odd term. So Adler was a disciple of Freud’s?
PHILOSOPHER: No, he was not. That misconception is common; we must dispel it. For one
thing, Adler and Freud were relatively close in age, and the relationship they formed as
researchers was founded upon equal footin

--- Chunk 2 ---
But I was struck then by a certain fact. What I was interested in was not solely Adlerian
psychology but rather something that had emerged through the  lter of the philosopher,
Ichiro Kishimi: It was Kishimi-Adler studies that I was seeking.
Grounded in the thought of Socrates and Plato and other an

--- Chunk 3 ---
philosophy, and that it is a proper  eld of study.
YOUTH: I have a passing knowledge of the psychology of Freud and Jung. A fascinating  eld.
PHILOSOPHER: Yes, Freud and Jung are both renowned. Adler was one of the original core
members of the Vienna Psychoanalytic Society, which was led by Freud. H

--- Chunk 4 ---
Authors’ Note
Sigmund Freud, C

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re
# Notice the corrected import here, plus RunnableParallel
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

# --- Helper: join retrieved chunks and scrub PDF null bytes ---
def format_docs(docs):
    raw_text = "\n\n---\n\n".join(doc.page_content for doc in docs)
    # This removes the invisible "End of String" assassin characters!
    clean_text = raw_text.replace("\x00", "").replace("\u0000", "")
    return clean_text

def strip_thinking(text: str) -> str:
    # re.DOTALL lets '.' match newlines too, since the thinking spans multiple lines
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

clean_output = RunnableLambda(strip_thinking)

SYSTEM_PROMPT = """
You are a helpful and precise AI assistant. Your primary task is to answer the user's question based strictly on the provided context.

Here are your rules:
1. Analyze the provided context chunks carefully.
2. Answer the question using ONLY the information found in the context.
3. If the context does not contain the answer, politely state: "I cannot answer this based on the provided text." Do not use outside knowledge or make up an answer.
4. Keep your answers concise and directly address the user's query.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# -- LLM via Groq API ---
llm = ChatGroq(
    model="qwen/qwen3-32b", # Swapped to a currently supported model
    temperature=0,
)

# --- THE CORRECTED LCEL CHAIN ---
chain = (
    # 1. Fetch the documents and pass the question through
    {"context": retriever, "question": RunnablePassthrough()}
    
    # 2. Assign the 'answer' key by formatting the docs and running the LLM
    | RunnablePassthrough.assign(
        answer=(
            {
                "context": lambda x: format_docs(x["context"]), 
                "question": lambda x: x["question"]
            }
            | prompt
            | llm
            | StrOutputParser()
            | clean_output          
        )
    )
)

print("RAG chain assembled perfectly.")

RAG chain assembled perfectly.


In [28]:
# --- Test it out ---
test_query = "Can you summarize what the philosopher says about trauma"
response = chain.invoke(test_query)

print("\n--- Answer ---")
print(response["answer"])

print("\n--- Sources Used ---")
for i, doc in enumerate(response["context"], 1):
    print(f"Chunk {i}: {doc.page_content[:150]}...")


--- Answer ---
<think>
Okay, let's see. The user is asking whether Adler was a disciple of Freud according to the philosopher in the context.

First, I need to look through the provided context for any mentions of Adler's relationship with Freud. 

In the first chunk, the Youth asks if Adler was a disciple of Freud. The Philosopher responds by saying, "No, he was not. That misconception is common; we must dispel it." They also mention that Adler and Freud were close in age and their relationship was on equal footing, unlike Jung who revered Freud as a father figure. 

So the key points here are that the philosopher explicitly denies that Adler was a disciple, pointing out their equal relationship. The user's question is directly addressed in this part. There's no conflicting information in the other chunks. The other parts talk more about Adler's individual psychology and Kishimi's influence, but not about his relationship with Freud. 

Therefore, the answer should be that according t